# Fase 3 & 4 — Performance gap e dimostrazione del bias

**Workshop SIIAM — Bias di genere nei dataset clinici**

In questo notebook partiamo da un modello di intelligenza artificiale che prevede
se un paziente in terapia intensiva sopravvivrà o no al ricovero.
Vogliamo rispondere a due domande cliniche:

1. Il modello funziona allo stesso modo su **uomini e donne**?
2. Se cambio la composizione dei dati con cui addestro il modello, **cambia il comportamento** del modello?

> **Nota per chi legge**: tutte le celle di codice producono tabelle e grafici.
> Non dovete scrivere codice: basta leggere gli output. Ogni sezione ha una
> breve spiegazione *prima* del codice e un commento *dopo* il grafico.

---

## Agenda

- **Parte A (Fase 3)** — Addestriamo 3 modelli (uno semplice, uno moderno, uno "deep learning")
  e misuriamo la loro performance separatamente su uomini e donne.
- **Parte B (Fase 4)** — Teniamo lo **stesso algoritmo** e cambiamo solo i dati di
  addestramento: vediamo che il bias cambia. Il test set resta **identico** in tutti gli scenari.



## 0 · Preparazione dell'ambiente

Carichiamo le librerie e i dati. Questa cella va eseguita una sola volta.

In [ ]:
# Montaggio Google Drive (solo su Google Colab)
from google.colab import drive
drive.mount('/content/drive')

# Installazione libreria XGBoost (già presente in Colab nella maggior parte dei casi)
!pip install xgboost -q

In [ ]:
# Librerie standard per analisi e grafici
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modelli di machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from xgboost import XGBClassifier

# Rete neurale (mini)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from IPython.display import display
import warnings; warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Colori del workshop — coerenti ovunque
COLOR_M       = '#1f77b4'   # uomini: blu
COLOR_F       = '#e74c3c'   # donne: rosso
COLOR_OVERALL = '#2ca02c'   # totale: verde

print('Librerie pronte.')

In [ ]:
# Percorso dei dati sul Drive — modificare se serve
BASE_PATH       = '/content/drive/MyDrive/Workshop_1/Workshop_Dataset/'
PRECOMPUTED_DIR = BASE_PATH + 'precomputed/'

# Interruttore da attivare IN AULA SOLO SE il training live va in crash.
#   False = addestriamo in diretta i modelli di Fase 4 (default)
#   True  = saltiamo il training e ricarichiamo i risultati già salvati su Drive
USE_PRECOMPUTED = False

import os
os.makedirs(PRECOMPUTED_DIR, exist_ok=True)

# I dati sono già stati puliti e standardizzati (media 0, deviazione 1) per ogni colonna.
# Il test set è fisso: NON lo tocchiamo mai.
train_features = pd.read_csv(BASE_PATH + 'train_features.csv')
train_labels   = pd.read_csv(BASE_PATH + 'train_labels.csv')
test_features  = pd.read_csv(BASE_PATH + 'test_features.csv')
test_labels    = pd.read_csv(BASE_PATH + 'test_labels.csv')

y_train = train_labels.values.ravel()
y_test  = test_labels.values.ravel()

print(f'Training: {train_features.shape[0]:,} pazienti, {train_features.shape[1]} variabili')
print(f'Test:     {test_features.shape[0]:,} pazienti')
print(f'Mortalità nel training: {y_train.mean()*100:.1f}%')
print(f'Mortalità nel test:     {y_test.mean()*100:.1f}%')

> **Cosa sta succedendo**: abbiamo **~8% di pazienti deceduti**. È un dataset
> molto **sbilanciato**. Questo dettaglio — spesso sottovalutato — cambia tutto
> quello che vedremo sotto, quindi tenetelo presente.
>
> Ricordatelo con questa immagine: se dico "nessuno morirà mai" indovino il 92% dei casi,
> ma non ho aiutato nessun paziente. Un'accuracy alta può mentire.

## 1 · Il linguaggio delle metriche, con un'analogia clinica

Tutto quello che faremo in questo notebook si riduce a pochi numeri che voi già conoscete
dai test diagnostici.

| Metrica | Domanda clinica equivalente |
|---|---|
| **Sensitivity** (= Recall) | *Se il paziente morirà davvero, il modello lo segnala?* |
| **Specificity** | *Se il paziente sopravviverà davvero, il modello lo lascia in pace?* |
| **Precision** | *Dei pazienti che il modello dice "morirà", quanti muoiono davvero?* |
| **AUC-ROC** | *Se pesco a caso un paziente deceduto e uno sopravvissuto, il modello assegna un rischio più alto al deceduto?* Valore tra 0.5 (caso) e 1 (perfetto). |
| **Accuracy** | *Su tutti i pazienti, quanti ne indovina?* Ingannevole su dati sbilanciati. |

Il modello produce una **probabilità** (es. 0.37 di morte).
Poi si sceglie una **soglia** per trasformarla in decisione binaria (es. ≥ 0.5 → "morirà").
La soglia 0.5 non è sacra: cambiarla sposta il trade-off tra sensitivity e specificity.
Oggi la teniamo a 0.5 per semplicità.

In [ ]:
def evaluate_model(model, X, y, is_torch=False):
    '''
    Restituisce una tabella con le metriche globali, poi per Maschi e per Femmine.
    Stesso codice per sklearn, xgboost e PyTorch: cambia solo come si fanno le predizioni.
    '''
    def predict(X_sub):
        if is_torch:
            model.eval()
            with torch.no_grad():
                logits = model(torch.FloatTensor(X_sub.values)).numpy().ravel()
            y_prob = 1 / (1 + np.exp(-logits))   # sigmoid
        else:
            y_prob = model.predict_proba(X_sub)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        return y_pred, y_prob

    def metrics(y_true, y_pred, y_prob):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return {
            'N': len(y_true),
            'Accuracy':   accuracy_score(y_true, y_pred),
            'Precision':  precision_score(y_true, y_pred, zero_division=0),
            'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
            'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
            'F1':         f1_score(y_true, y_pred, zero_division=0),
            'AUC-ROC':    roc_auc_score(y_true, y_prob),
        }

    mask_M = X['gender_M'] == 1
    mask_F = X['gender_F'] == 1

    y_pred_all, y_prob_all = predict(X)
    y_pred_M,   y_prob_M   = predict(X[mask_M])
    y_pred_F,   y_prob_F   = predict(X[mask_F])

    out = pd.DataFrame({
        'Totale':   metrics(y,         y_pred_all, y_prob_all),
        'Maschi':   metrics(y[mask_M], y_pred_M,   y_prob_M),
        'Femmine':  metrics(y[mask_F], y_pred_F,   y_prob_F),
    }).T
    out['N'] = out['N'].astype(int)
    return out.round(3)

print('Funzione evaluate_model pronta.')

## 2 · Baseline 1 — Regressione Logistica

La **regressione logistica** è il modello di ML più vecchio e semplice: combina le
variabili del paziente con dei pesi e produce una probabilità tra 0 e 1.
È lo stesso principio dei punteggi clinici tipo APACHE o SOFA, ma con i pesi
appresi automaticamente dai dati.

Lo addestriamo prima in modo "ingenuo" per far vedere un problema molto comune.

In [ ]:
lr_naive = LogisticRegression(max_iter=1000, random_state=42)
lr_naive.fit(train_features, y_train)

print('Modello ingenuo addestrato. Ora vediamo come si comporta per sesso.')
lr_naive_results = evaluate_model(lr_naive, test_features, y_test)
lr_naive_results

> **Guardate la colonna `Sensitivity`**: il modello ingenuo "prende" pochissimi
> dei pazienti che moriranno davvero. Perché? Perché i deceduti sono solo l'8%:
> il modello impara che conviene dire "sopravviverà" quasi sempre.
> Questo è **l'effetto dello sbilanciamento di classe**.
>
> Soluzione didattica standard: dare più peso ai casi rari durante l'addestramento
> (`class_weight='balanced'`). Rifacciamo il modello così.

In [ ]:
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(train_features, y_train)

lr_results = evaluate_model(lr_model, test_features, y_test)
lr_results

Ora la sensitivity è molto più alta: il modello prova davvero a cogliere i
pazienti a rischio. In cambio perde un po' di specificity (alcuni allarmi falsi).
Questo è un **trade-off**, non un errore.

Confrontiamolo a colpo d'occhio su maschi e femmine.

In [ ]:
def plot_M_vs_F(results, title):
    metrics_show = ['AUC-ROC', 'Sensitivity', 'Specificity', 'Precision']
    vals_M = results.loc['Maschi',  metrics_show].values
    vals_F = results.loc['Femmine', metrics_show].values

    x = np.arange(len(metrics_show)); w = 0.35
    fig, ax = plt.subplots(figsize=(9, 4.5))
    b1 = ax.bar(x - w/2, vals_M, w, label='Maschi',  color=COLOR_M, alpha=.9)
    b2 = ax.bar(x + w/2, vals_F, w, label='Femmine', color=COLOR_F, alpha=.9)
    for b in list(b1) + list(b2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.01,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(metrics_show)
    ax.set_ylim(0, 1.05); ax.set_ylabel('Valore')
    ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

plot_M_vs_F(lr_results, 'Regressione Logistica — Maschi vs Femmine')

Già qui si può osservare una prima **differenza fra i due sessi**, specialmente sull'AUC.
È il nostro primo esempio di **performance gap**. Torniamo sul punto più avanti.

In [ ]:
# Matrici di confusione M vs F — mostriamo ANCHE le percentuali, non solo i conteggi.
mask_M_test = test_features['gender_M'] == 1
mask_F_test = test_features['gender_F'] == 1
y_pred_lr = lr_model.predict(test_features)

def cm_pct(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    ann = np.array([[f'{c}\\n({c/cm.sum()*100:.1f}%)' for c in row] for row in cm])
    return cm, ann

cm_m, ann_m = cm_pct(y_test[mask_M_test], y_pred_lr[mask_M_test])
cm_f, ann_f = cm_pct(y_test[mask_F_test], y_pred_lr[mask_F_test])

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
vmax = max(cm_m.max(), cm_f.max())
for ax, cm, ann, title, cmap in [
    (axes[0], cm_m, ann_m, f'Maschi  (n={mask_M_test.sum():,})', 'Blues'),
    (axes[1], cm_f, ann_f, f'Femmine (n={mask_F_test.sum():,})', 'Reds'),
]:
    sns.heatmap(cm, annot=ann, fmt='', cmap=cmap, ax=ax, vmin=0, vmax=vmax,
                xticklabels=['Pred: Vivo', 'Pred: Deceduto'],
                yticklabels=['Reale: Vivo', 'Reale: Deceduto'])
    ax.set_title(title)
plt.suptitle('Matrici di confusione (LR, soglia 0.5)', y=1.02); plt.tight_layout(); plt.show()

## 3 · Baseline 2 — XGBoost (ensemble di alberi decisionali)

XGBoost è un algoritmo più moderno: costruisce **tanti piccoli alberi decisionali**
in sequenza, ognuno che corregge gli errori del precedente.
Analogia clinica: è come chiedere il parere a 100 specialisti che si passano le
cartelle, ognuno aggiungendo un pezzo di ragionamento.

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, eval_metric='logloss')
xgb_model.fit(train_features, y_train)

# Valutiamo sia sul training che sul test: confronto utile per vedere l'overfitting
xgb_train_results = evaluate_model(xgb_model, train_features, y_train)
xgb_results       = evaluate_model(xgb_model, test_features,  y_test)

print('— Performance sul TRAINING (i pazienti che il modello ha già visto) —')
display(xgb_train_results)
print('— Performance sul TEST (pazienti nuovi) —')
xgb_results

> **Overfitting**: sul training XGBoost raggiunge metriche quasi perfette,
> sul test cala. Significa che il modello **impara a memoria** i pazienti
> già visti, ma generalizza peggio sui nuovi. Morale della storia:
> guardate sempre i numeri **sul test**, mai sul training.

In [ ]:
plot_M_vs_F(xgb_results, 'XGBoost — Maschi vs Femmine (solo test)')

## 4 · Mini-MLP — una piccola rete neurale

Un **MLP** (Multi-Layer Perceptron) è la forma più semplice di "deep learning":
alcuni strati di neuroni artificiali che combinano le variabili a più livelli di astrazione.
Ne usiamo uno volutamente piccolo — l'obiettivo è **mostrare che un modello più
sofisticato non risolve il problema per magia**.

In [ ]:
class MiniMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32),          nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1),   # output "logit" — NO sigmoid qui
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

# Dati → tensori PyTorch
X_tr = torch.FloatTensor(train_features.values)
y_tr = torch.FloatTensor(y_train)
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=512, shuffle=True)

# Gestione sbilanciamento con un solo loss, pulito
pos_weight = torch.tensor([(len(y_train) - y_train.sum()) / y_train.sum()])
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

torch.manual_seed(42)
mlp_model = MiniMLP(train_features.shape[1])
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

n_epochs, losses = 10, []
for epoch in range(n_epochs):
    mlp_model.train(); running = 0.0; n_batches = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(mlp_model(xb), yb)
        loss.backward(); optimizer.step()
        running += loss.item(); n_batches += 1
    losses.append(running / n_batches)
    if (epoch + 1) % 2 == 0:
        print(f'Epoca {epoch+1}/{n_epochs}  —  loss {losses[-1]:.4f}')

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, n_epochs+1), losses, 'o-', color='purple')
ax.set_xlabel('Epoca'); ax.set_ylabel('Loss (BCE)')
ax.set_title('Andamento dell\'addestramento della mini-MLP')
plt.tight_layout(); plt.show()

In [ ]:
mlp_results = evaluate_model(mlp_model, test_features, y_test, is_torch=True)
mlp_results

In [ ]:
plot_M_vs_F(mlp_results, 'Mini-MLP — Maschi vs Femmine')

## 5 · Confronto dei 3 modelli — il messaggio chiave di Fase 3

Sovrapponiamo i risultati dei 3 modelli sullo stesso grafico, separatamente
per le tre metriche più rilevanti in clinica.

In [ ]:
models = ['Regressione Logistica', 'XGBoost', 'Mini-MLP']
res_all = [lr_results, xgb_results, mlp_results]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric in zip(axes, ['AUC-ROC', 'Sensitivity', 'Specificity']):
    vals_M = [r.loc['Maschi',  metric] for r in res_all]
    vals_F = [r.loc['Femmine', metric] for r in res_all]
    x = np.arange(len(models)); w = 0.35
    b1 = ax.bar(x - w/2, vals_M, w, label='Maschi',  color=COLOR_M, alpha=.9)
    b2 = ax.bar(x + w/2, vals_F, w, label='Femmine', color=COLOR_F, alpha=.9)
    for b in list(b1) + list(b2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.005,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(models, fontsize=9, rotation=10)
    ax.set_title(metric); ax.set_ylim(0, 1.05); ax.legend()
plt.suptitle('Performance per sesso nei 3 modelli', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Tabella riassuntiva compatta: solo AUC e Sensitivity, con il gap M-F
def summary_row(name, res):
    return {
        'Modello': name,
        'AUC M':  res.loc['Maschi',  'AUC-ROC'],
        'AUC F':  res.loc['Femmine', 'AUC-ROC'],
        'Gap AUC (M-F)':  round(res.loc['Maschi','AUC-ROC'] - res.loc['Femmine','AUC-ROC'], 3),
        'Sens M': res.loc['Maschi',  'Sensitivity'],
        'Sens F': res.loc['Femmine', 'Sensitivity'],
        'Gap Sens (M-F)': round(res.loc['Maschi','Sensitivity'] - res.loc['Femmine','Sensitivity'], 3),
    }

summary_fase3 = pd.DataFrame([
    summary_row('Regressione Logistica', lr_results),
    summary_row('XGBoost',               xgb_results),
    summary_row('Mini-MLP',              mlp_results),
]).set_index('Modello')
summary_fase3

### Messaggio di Fase 3

> - Il gap di performance fra uomini e donne **è presente in tutti e tre i modelli**.
> - Il modello più sofisticato (Mini-MLP) **non annulla il divario**.
> - Le metriche **globali** (colonna Totale) possono sembrare buone, ma nascondono
>   differenze importanti quando si stratifica per sesso.
>
> Questa è la prima dimostrazione concreta per i clinici: *un modello più moderno
> non risolve automaticamente il problema di fairness*.

> **Nota operativa per il giorno del workshop**
>
> Le celle che seguono addestrano modelli in diretta. In condizioni normali ci vogliono
> 2–3 minuti su Colab. Se la connessione o l'ambiente vanno in crash e non si riesce a
> rifare il training in aula, basta:
>
> 1. tornare alla cella di setup all'inizio del notebook,
> 2. cambiare `USE_PRECOMPUTED = False` in `USE_PRECOMPUTED = True`,
> 3. rilanciare da lì in poi.
>
> Il notebook leggerà i risultati dalla cartella `precomputed/` su Drive, salvata
> durante l'ultima esecuzione completa. **Dopo ogni modifica ricordarsi di ri-eseguire
> almeno una volta in modalità live per aggiornare i file salvati.**

---

## 6 · Fase 4 — Stessa ricetta, ingredienti diversi

Adesso facciamo l'esperimento che **è il cuore del workshop**:

> *A parità di algoritmo, cambio solo la **composizione dei dati di addestramento**
> e guardo cosa succede al bias.*

**Vincolo non negoziabile**: il **test set è fisso e identico in tutti gli scenari**.
Se cambiassimo anche il test, non sapremmo dire se le differenze arrivano dal
modello o dalla valutazione.

### Proxy di severità clinica: `gcs_motor_apache`

Per definire un paziente "più grave" o "meno grave" usiamo la componente motoria
della **Glasgow Coma Scale** (GCS motor), che conoscete bene: valori bassi =
paziente compromesso. Le variabili nel dataset sono già standardizzate
(z-score), quindi:

- `gcs_motor_apache < 0`  →  **severità alta** (sotto la mediana)
- `gcs_motor_apache ≥ 0`  →  **severità bassa**

### Gli scenari

| Scenario | Sesso nel training | Severità nel training |
|---|---|---|
| **Originale** | ~54% M / ~46% F (invariato) | invariato |
| **A** | ~54% M / ~46% F (invariato) | quasi **solo pazienti gravi** (90/10) |
| **B** | ~54% M / ~46% F (invariato) | **bilanciato** (50/50) |

In [ ]:
# 1) Etichettiamo severità nei dati originali
sev_train = (train_features['gcs_motor_apache'] < 0).astype(int)  # 1 = grave
sev_test  = (test_features['gcs_motor_apache']  < 0).astype(int)

print('Distribuzione severità nel training originale:')
print(f'  Gravi (severità alta): {(sev_train==1).sum():,} ({(sev_train==1).mean()*100:.1f}%)')
print(f'  Non gravi:             {(sev_train==0).sum():,} ({(sev_train==0).mean()*100:.1f}%)')
print()
print('Mortalità osservata per livello di severità:')
print(f'  Gravi:     {y_train[sev_train==1].mean()*100:.1f}%')
print(f'  Non gravi: {y_train[sev_train==0].mean()*100:.1f}%')

In [ ]:
def build_scenario(target_pct_severe, label, seed=42):
    '''Costruisce un training set dove i pazienti "gravi" sono il target_pct_severe% del totale.
       La proporzione uomini/donne resta invariata: non tocchiamo il sesso.'''
    rng = np.random.default_rng(seed)
    idx_severe   = np.where(sev_train == 1)[0]
    idx_nsevere  = np.where(sev_train == 0)[0]

    # Sottocampioniamo la categoria sovra-rappresentata rispetto al target
    n_sev, n_nsv = len(idx_severe), len(idx_nsevere)
    # risolvo n_severe / (n_severe + n_nsevere) = target_pct_severe
    # teniamo tutti quelli disponibili nel gruppo minoritario desiderato
    if target_pct_severe >= n_sev / (n_sev + n_nsv):
        # vogliamo più gravi del presente → limitiamo i non-gravi
        k_severe = n_sev
        k_nsv    = int(k_severe * (1 - target_pct_severe) / target_pct_severe)
        k_nsv    = min(k_nsv, n_nsv)
    else:
        # vogliamo meno gravi → limitiamo i gravi
        k_nsv    = n_nsv
        k_severe = int(k_nsv * target_pct_severe / (1 - target_pct_severe))
        k_severe = min(k_severe, n_sev)

    pick_s  = rng.choice(idx_severe,  size=k_severe, replace=False)
    pick_ns = rng.choice(idx_nsevere, size=k_nsv,    replace=False)
    idx     = np.concatenate([pick_s, pick_ns]); rng.shuffle(idx)

    X = train_features.iloc[idx].reset_index(drop=True)
    y = y_train[idx]
    pct_sev = (sev_train.iloc[idx] == 1).mean() * 100
    pct_F   = (X['gender_F'] == 1).mean() * 100
    print(f'[{label}] N={len(X):,}  |  Gravi={pct_sev:.1f}%  |  Donne={pct_F:.1f}%')
    return X, y

# Scenario A — quasi solo pazienti gravi
X_A, y_A = build_scenario(target_pct_severe=0.90, label='Scenario A (gravi 90%)')
# Scenario B — severità bilanciata
X_B, y_B = build_scenario(target_pct_severe=0.50, label='Scenario B (gravi 50%)')

Ora addestriamo lo **stesso** XGBoost sui tre scenari (Originale / A / B).
Per essere sicuri che le differenze che vediamo **non siano rumore casuale**,
rifacciamo tutto con **3 seed diversi** e riportiamo la media ± deviazione standard.

In [ ]:
scenarios = {
    'Originale':     (train_features, y_train),
    'A (gravi 90%)': (X_A, y_A),
    'B (gravi 50%)': (X_B, y_B),
}

def _slug(name):
    return (name.replace(' ', '_').replace('(', '').replace(')', '')
                .replace('%', 'pct').replace('/', '_'))

mean_results, std_results = {}, {}

if USE_PRECOMPUTED:
    print('Modalità PRE-CALCOLATA attiva: carico i risultati da Drive.')
    for name in scenarios:
        s = _slug(name)
        mean_results[name] = pd.read_csv(PRECOMPUTED_DIR + f'fase4_{s}_mean.csv', index_col=0)
        std_results[name]  = pd.read_csv(PRECOMPUTED_DIR + f'fase4_{s}_std.csv',  index_col=0)
        print(f'  caricato: {name}')
else:
    for name, (X, y) in scenarios.items():
        print(f'Addestro scenario: {name}')
        mean_results[name], std_results[name] = train_and_eval(X, y)
    # Salvo i risultati su Drive per poterli riusare senza re-training
    for name in mean_results:
        s = _slug(name)
        mean_results[name].to_csv(PRECOMPUTED_DIR + f'fase4_{s}_mean.csv')
        std_results[name].to_csv( PRECOMPUTED_DIR + f'fase4_{s}_std.csv')
    print(f'\nFatto. Risultati salvati in: {PRECOMPUTED_DIR}')

In [ ]:
# Mostriamo le 3 tabelle medie affiancate (solo le metriche chiave)
for name in scenarios:
    print(f'\n=== {name} — media su 3 seed ===')
    display(mean_results[name][['Accuracy', 'Sensitivity', 'Specificity', 'AUC-ROC']])

In [ ]:
# Grafico comparativo: barre con errore (std su 3 seed) per Maschi e Femmine
# Mettiamo affiancate AUC, Sensitivity, Specificity
scenario_names = list(scenarios.keys())
metrics_show   = ['AUC-ROC', 'Sensitivity', 'Specificity']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, metric in zip(axes, metrics_show):
    vals_M  = [mean_results[s].loc['Maschi',  metric] for s in scenario_names]
    errs_M  = [std_results[s].loc['Maschi',   metric] for s in scenario_names]
    vals_F  = [mean_results[s].loc['Femmine', metric] for s in scenario_names]
    errs_F  = [std_results[s].loc['Femmine',  metric] for s in scenario_names]

    x = np.arange(len(scenario_names)); w = 0.35
    b1 = ax.bar(x - w/2, vals_M, w, yerr=errs_M, capsize=4,
                label='Maschi',  color=COLOR_M, alpha=.9)
    b2 = ax.bar(x + w/2, vals_F, w, yerr=errs_F, capsize=4,
                label='Femmine', color=COLOR_F, alpha=.9)
    for b in list(b1) + list(b2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+.01,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(scenario_names, fontsize=9)
    ax.set_title(metric); ax.set_ylim(0, 1.05); ax.legend()
plt.suptitle('Fase 4 — Stesso algoritmo, training diverso. Test set FISSO.',
             y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Un solo grafico "riassunto": il GAP M-F di AUC, per ciascuno scenario
gap_auc  = [mean_results[s].loc['Maschi','AUC-ROC']    - mean_results[s].loc['Femmine','AUC-ROC']    for s in scenario_names]
gap_sens = [mean_results[s].loc['Maschi','Sensitivity']- mean_results[s].loc['Femmine','Sensitivity'] for s in scenario_names]
gap_spec = [mean_results[s].loc['Maschi','Specificity']- mean_results[s].loc['Femmine','Specificity'] for s in scenario_names]

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(scenario_names)); w = 0.25
ax.bar(x - w, gap_auc,  w, label='Gap AUC',         color='#4C72B0')
ax.bar(x,     gap_sens, w, label='Gap Sensitivity', color='#DD8452')
ax.bar(x + w, gap_spec, w, label='Gap Specificity', color='#55A868')
ax.axhline(0, color='k', linewidth=.8)
ax.set_xticks(x); ax.set_xticklabels(scenario_names)
ax.set_ylabel('Differenza M − F (positivo = meglio sui maschi)')
ax.set_title('Gap di performance fra sessi al variare del training set')
ax.legend(); plt.tight_layout(); plt.show()

### Lettura del grafico (Fase 4)

> - Il test set è **lo stesso in tutti gli scenari**: qualunque differenza che
>   vedete è colpa del **training**.
> - Scenario A (solo gravi nel training) e Scenario B (gravi/non-gravi bilanciati)
>   producono due modelli **molto diversi** sul medesimo test, con gap di performance
>   fra maschi e femmine diversi — anche se il sesso **non è stato toccato**.
> - La barra d'errore dei grafici mostra che la differenza fra scenari **non è rumore**.
>
> **Conclusione cardine del workshop**: il bias che osserviamo non nasce
> dall'algoritmo, ma dal **dataset**. Cambiare il modo in cui raccogliamo e
> selezioniamo i dati cambia il comportamento del modello sui sottogruppi,
> anche senza toccare il sottogruppo stesso.

## 7 · Tabella riassuntiva finale e take-home messages

In [ ]:
final_rows = []
# Fase 3: 3 modelli
for name, r in [('LR',  lr_results), ('XGB', xgb_results), ('MLP', mlp_results)]:
    final_rows.append({
        'Blocco': 'Fase 3', 'Setup': name,
        'AUC M': r.loc['Maschi','AUC-ROC'], 'AUC F': r.loc['Femmine','AUC-ROC'],
        'Gap AUC': round(r.loc['Maschi','AUC-ROC'] - r.loc['Femmine','AUC-ROC'], 3),
        'Sens M': r.loc['Maschi','Sensitivity'], 'Sens F': r.loc['Femmine','Sensitivity'],
        'Gap Sens': round(r.loc['Maschi','Sensitivity'] - r.loc['Femmine','Sensitivity'], 3),
    })
# Fase 4: 3 scenari
for name in scenario_names:
    r = mean_results[name]
    final_rows.append({
        'Blocco': 'Fase 4', 'Setup': f'XGB — {name}',
        'AUC M': r.loc['Maschi','AUC-ROC'], 'AUC F': r.loc['Femmine','AUC-ROC'],
        'Gap AUC': round(r.loc['Maschi','AUC-ROC'] - r.loc['Femmine','AUC-ROC'], 3),
        'Sens M': r.loc['Maschi','Sensitivity'], 'Sens F': r.loc['Femmine','Sensitivity'],
        'Gap Sens': round(r.loc['Maschi','Sensitivity'] - r.loc['Femmine','Sensitivity'], 3),
    })

final_table = pd.DataFrame(final_rows)
final_table

### Take-home messages per i partecipanti

1. Un dataset "bilanciato per sesso" **non è automaticamente rappresentativo**
   e non garantisce performance equivalenti.
2. **L'accuracy globale mente**: su dati sbilanciati bisogna sempre stratificare
   per sottogruppo e guardare **sensitivity e specificity separatamente**.
3. **Più moderno ≠ più equo**: LR, XGBoost e Mini-MLP mostrano tutti un gap.
4. Il bias **nasce nel dataset**, non nell'algoritmo: cambiando la composizione
   del training cambia il comportamento osservato nei sottogruppi — anche se
   il sottogruppo (il sesso, qui) **non è stato toccato**.

---

## Appendice — analisi extra (non per la presentazione live)

Queste sezioni sono lasciate per chi volesse approfondire sul notebook dopo
il workshop. **Non vanno mostrate nella lezione frontale** per non appesantire.

### A.1 · Analisi intersezionale Genere × Etnia
### A.2 · Rimozione progressiva delle donne dal training (replica HW1 di Luigi)

In [ ]:
# A.1 — Intersezione Genere x Etnia sul modello XGBoost base
def eval_subgroup(model, X, y, mask, name):
    Xs, ys = X[mask], y[mask]
    yp = model.predict(Xs)
    yprob = model.predict_proba(Xs)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ys, yp, labels=[0,1]).ravel()
    return {'Sottogruppo': name, 'N': int(len(ys)),
            'Sens': round(recall_score(ys, yp, zero_division=0), 3),
            'Spec': round(tn / (tn + fp) if (tn + fp) > 0 else 0, 3),
            'AUC':  round(roc_auc_score(ys, yprob), 3)}

m_M  = test_features['gender_M'] == 1
m_F  = test_features['gender_F'] == 1
m_C  = test_features['ethnicity_Caucasian'] == 1
m_NC = test_features['ethnicity_Caucasian'] == 0

inter = pd.DataFrame([
    eval_subgroup(xgb_model, test_features, y_test, m_M & m_C,  'M / Caucasico'),
    eval_subgroup(xgb_model, test_features, y_test, m_M & m_NC, 'M / Non-Caucasico'),
    eval_subgroup(xgb_model, test_features, y_test, m_F & m_C,  'F / Caucasico'),
    eval_subgroup(xgb_model, test_features, y_test, m_F & m_NC, 'F / Non-Caucasico'),
]).set_index('Sottogruppo')
inter

In [ ]:
# A.2 — Rimozione progressiva delle donne (0 / 40 / 80%) con 3 seed per scenario
removal_pcts = [0, 40, 80]
rem_mean, rem_std = {}, {}
idx_F_tr = np.where(train_features['gender_F'] == 1)[0]

if USE_PRECOMPUTED:
    print('Modalità PRE-CALCOLATA attiva: carico appendice A.2 da Drive.')
    for pct in removal_pcts:
        rem_mean[pct] = pd.read_csv(PRECOMPUTED_DIR + f'appA2_rem{pct}_mean.csv', index_col=0)
        rem_std[pct]  = pd.read_csv(PRECOMPUTED_DIR + f'appA2_rem{pct}_std.csv',  index_col=0)
else:
    for pct in removal_pcts:
        accs = []
        for seed in (42, 7, 123):
            rng = np.random.default_rng(seed)
            if pct == 0:
                Xs, ys = train_features, y_train
            else:
                n_rm = int(len(idx_F_tr) * pct / 100)
                rm   = rng.choice(idx_F_tr, size=n_rm, replace=False)
                keep = np.setdiff1d(np.arange(len(train_features)), rm)
                Xs, ys = train_features.iloc[keep], y_train[keep]
            m = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                              random_state=seed, eval_metric='logloss')
            m.fit(Xs, ys)
            accs.append(evaluate_model(m, test_features, y_test))
        rem_mean[pct] = sum(accs) / len(accs)
        rem_std[pct]  = pd.concat([a.stack() for a in accs], axis=1).std(axis=1).unstack()
        print(f'Rimozione {pct}%: fatto')
    # Salvo su Drive
    for pct in removal_pcts:
        rem_mean[pct].to_csv(PRECOMPUTED_DIR + f'appA2_rem{pct}_mean.csv')
        rem_std[pct].to_csv( PRECOMPUTED_DIR + f'appA2_rem{pct}_std.csv')

In [ ]:
# Grafico: AUC per genere al variare della % di donne rimosse (con banda di errore)
auc_M = [rem_mean[p].loc['Maschi',  'AUC-ROC'] for p in removal_pcts]
auc_F = [rem_mean[p].loc['Femmine', 'AUC-ROC'] for p in removal_pcts]
err_M = [rem_std[p].loc['Maschi',   'AUC-ROC'] for p in removal_pcts]
err_F = [rem_std[p].loc['Femmine',  'AUC-ROC'] for p in removal_pcts]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.errorbar(removal_pcts, auc_M, yerr=err_M, fmt='s-', color=COLOR_M,
            capsize=4, label='Maschi',  linewidth=2, markersize=8)
ax.errorbar(removal_pcts, auc_F, yerr=err_F, fmt='^-', color=COLOR_F,
            capsize=4, label='Femmine', linewidth=2, markersize=8)
ax.set_xticks(removal_pcts)
ax.set_xlabel('% donne rimosse dal training')
ax.set_ylabel('AUC-ROC sul test set (fisso)')
ax.set_title('Togliere donne dal training peggiora le donne sul test')
ax.legend(); plt.tight_layout(); plt.show()